# Road Damage Inspection System

**Team:** Member 1 — Austin Wang · Member 2 — Kevin Fan · Member 3 — RJ Xia

## 8:17 a.m. — one road image, three urgent questions

> **What broke? Where is it? How much road surface is affected?**

<p align="center"><img src="artifacts/final/marked_pothole_hook.jpg" alt="A road pothole circled in white paint for repair" width="920"></p>

An inspection vehicle may see a road only once, while a city may have thousands of frames waiting. The project turns each frame into visible evidence for an inspector—not an autonomous repair order.

**Project in one sentence:** road pixels → boxes + masks → explained priority → human judgment.


Municipal road inspection turns a large image stream into three practical questions: what type of damage is present, where is it located, and how much surface is affected? Detection, segmentation, and a transparent priority rule organize the available visual evidence. The result is intended for triage rather than repair authorization, so a human reviewer remains responsible for the final decision.

# One image, two specialists, one reviewable decision

<p align="center"><img src="artifacts/presentation/project_workflow.svg" alt="End-to-end project workflow from data and road image to two models, evidence fusion, and human review" width="1100"></p>

**Takeaway:** YOLO answers **what and where**; SegFormer answers **how much area**. Their outputs are fused only to explain and rank cases for human review.


The workflow has two parallel vision branches. YOLO11s assigns a damage class, location, and confidence; SegFormer-B0 estimates the pothole footprint at pixel level. Green arrows identify the task-specific training sources, while the blue path follows one new road image through inference. Both outputs meet in a review interface, but the decision boundary remains with the inspector.

# A bounding box and a mask answer different questions

| Evidence | What the audience can see | What it cannot prove alone |
|---|---|---|
| **Detection box** | Damage class, approximate location, confidence | Exact damaged surface or depth |
| **Segmentation mask** | Pixel-level pothole footprint and area | Crack category or engineering severity |
| **Combined view** | Type, location, footprint, confidence, and reasons | A certified maintenance decision |

**Takeaway:** detection and segmentation are complementary measurements; neither output should be mistaken for a complete road-condition assessment.

Bounding boxes provide approximate location and class evidence, but the rectangle also contains undamaged pixels. A segmentation mask gives finer geometric information by classifying individual pixels, although this project’s mask model is binary and limited to potholes. Combining the outputs improves interpretability without creating new ground truth: depth, structural condition, traffic risk, and repair cost remain outside the scope of a single image.

# What do D00, D10, D20, and D40 mean?

RDD2022 uses four detection codes, each paired with a bounding box:

| Code | Meaning | Visual cue |
|---|---|---|
| **D00** | Longitudinal crack | Runs along the road |
| **D10** | Transverse crack | Crosses the road |
| **D20** | Alligator crack | Connected web of cracks |
| **D40** | Pothole | Local broken/depressed pavement |

<p align="center"><img src="artifacts/member1/EDA_Figures/09_damage_class_vocabulary.png" alt="RDD2022 examples of D00, D10, D20, and D40 with bounding boxes" width="960"></p>

**Takeaway:** the codes identify **damage type, not severity**—D40 is not automatically “level 40” damage.


RDD2022 defines four target categories. D00 runs along the road, D10 crosses it, D20 forms a connected alligator-like network, and D40 marks a localized pothole. Similar road textures can still belong to different categories, which makes the annotated examples important. These codes describe damage type rather than severity; a D40 label alone does not establish urgency.

# Two public datasets, two complementary label types

| Data source | What it contains | Labels | Project role |
|---|---:|---|---|
| **RDD2022** | 47,420 images; 38,385 publicly labeled | 55,006 valid target boxes across six countries / seven capture domains | Four-class object detection |
| **Pothole Mix** | 4,340 image-mask pairs from six component sources | Pixel masks; official 3,340 / 496 / 504 split | Binary pothole segmentation |

<p align="center"><img src="artifacts/member3/eda/m3_fig5_aligned_samples.png" alt="Aligned Pothole Mix road images and segmentation masks" width="900"></p>

**Takeaway:** these datasets are related road imagery but solve different cognitive problems: RDD2022 supplies **boxes and types**, while Pothole Mix supplies **pixel-level pothole shape**.


The two datasets provide complementary forms of supervision. RDD2022 contributes 47,420 images across six countries and seven capture domains, including 38,385 images with public bounding boxes. Pothole Mix contributes 4,340 paired images and pixel masks. The first dataset supports damage type and location, while the second supports pothole shape and surface coverage; a box is not treated as a substitute for a precise mask.

# Not every official image is supervised training data

<p align="center"><img src="artifacts/member1/EDA_Figures/00_all_annotation_code_counts.png" alt="All annotation codes found in the official RDD2022 XML files" width="940"></p>

| RDD2022 record type | Count | Project use |
|---|---:|---|
| Official images inventoried | 47,420 | File and image-level audit |
| Images with public XML | 38,385 | Supervised detection pool |
| Official-test images without public XML | 9,035 | Image-level EDA only |
| Valid target-negative images | 14,618 | Retained as background examples |

**Takeaway:** missing public ground truth is not the same as “no damage,” and non-target codes are audited rather than silently remapped.

RDD2022 contains more annotation codes than the four used by this project. Its 65,712 raw XML boxes include 10,705 non-target annotations such as D44, D50, repair, and region-specific labels. Those records remain in the audit trail but are not forced into the target vocabulary. The 9,035 official-test images also remain separate because the absence of public XML prevents supervised accuracy measurement.

# Member 1: make the data trustworthy before modeling

| Full-data audit | Verified result | Decision |
|---|---:|---|
| Official RDD2022 images decoded | 47,420 / 47,420 | No corrupt image removal |
| Publicly labeled images retained | 38,385 | Use the complete labeled pool |
| Raw XML boxes / target boxes | 65,712 / 55,007 | Audit 10,705 non-target codes; do not remap them |
| Exportable target boxes | 55,006 | Exclude one degenerate D20 box |
| Exact / exact+near duplicate rows | 4 / 3,256 | Retain but group-lock to one split |
| Valid target negatives | 14,618 images | Retain to teach background |

**Method chain:** Pascal VOC XML → image/box tables → integrity and duplicate audit → group-aware split → synchronized YOLO + COCO exports.

**Takeaway:** Member 1’s main product is not just EDA—it is a fixed, leakage-aware data contract shared by both detectors.


Part A converts the raw RDD2022 release into a stable modeling contract. All 47,420 images decode successfully, every publicly labeled image remains in scope, and target and non-target XML codes are audited separately. Exact and near-duplicate scenes are group-locked to one split, and only one degenerate target box is excluded from export. Synchronized YOLO and COCO outputs let later experiments trace results to the same image, split, and label versions.

# Three data-quality choices protect the experiment

| Risk | Tempting shortcut | Project decision |
|---|---|---|
| Exact and near duplicates | Randomly split every file | Lock each duplicate group to one split |
| Images with no target boxes | Delete them as “empty” | Keep valid negatives to teach background |
| One degenerate D20 box | Export it anyway | Exclude the invalid box and document it |

**Takeaway:** data cleaning is not simply deletion; each decision must preserve useful information while preventing leakage or malformed supervision.

A visually clean dataset can still yield misleading evaluation. Near-duplicate frames across training and test reward scene recognition rather than generalization. Removing all target-negative images deprives the detector of normal pavement and can increase false alarms, while a zero-area box creates an invalid training target. Group locking, negative retention, and one documented box exclusion address these risks without discarding the labeled pool.

# EDA prediction: small and rare damage will be the hard case

<p align="center"><img src="artifacts/member1/EDA_Figures/01_counts_country_and_class.png" alt="Image counts by country and target boxes by class" width="1040"></p>

<p style="text-align:center"><em>Top row: country imbalance and target-class imbalance.</em></p>

<p align="center"><img src="artifacts/member1/EDA_Figures/04_box_area_violin_and_center_heatmap.png" alt="Relative box area by class and object-center heatmap" width="1040"></p>

<p style="text-align:center"><em>Bottom row: object scale by class and spatial concentration of annotations.</em></p>

- Japan contributes 10,506 labeled images versus 2,829 from Czech.
- D00 supplies 47.3% of target boxes; D40 supplies only 11.9%.
- The median target occupies 1.43% of the image; D40 has the smallest class median (0.54%).

**Takeaway:** imbalance and tiny targets mean that overall mAP is insufficient; per-class recall—especially for potholes—must also be reported.

The EDA identifies two dominant risks before model training: imbalance and scale. Japan contributes 10,506 labeled images compared with 2,829 from Czech, while D00 accounts for 47.3% of target boxes and D40 only 11.9%. The median target covers 1.43% of an image, and the median D40 box only 0.54%. These distributions make per-class recall and small-object performance more informative than a single overall mAP value.

# The road domain changes beyond the damage label

<p align="center"><img src="artifacts/member1/EDA_Figures/08_visual_outliers.png" alt="RDD2022 examples with unusual lighting, blur, framing, snow, and camera viewpoints" width="980"></p>

- Drone frames may contain black borders and a near-vertical viewpoint.
- Underpasses, glare, snow, shadows, and windshield blur change local contrast.
- Country and capture device change resolution, road texture, markings, and scale.

**Takeaway:** a model must recognize damage across acquisition conditions, not only memorize the visual style of one camera or country.

The outliers are valid observations rather than corrupt files. Removing them would simplify the benchmark while reducing realism. Drone borders, distant dashboard views, snow, glare, darkness, and blur can make the same damage occupy very different numbers of pixels. This variation motivates performance slices by object size, geography, brightness, and blur, as well as the separate held-out-country experiment.

# From EDA to a fair evaluation design

<p align="center"><img src="artifacts/member1/EDA_Figures/05_resolution_brightness_contrast_blur.png" alt="Resolution, brightness, contrast, and blur distributions by country" width="930"></p>

| Leakage-safe split | Images |
|---|---:|
| Train | 26,888 |
| Validation | 5,714 |
| Test | 5,783 |
| Held-out-US auxiliary test | 4,805 |

Norway’s median resolution is 8.22 MP while most domains are roughly 0.26–0.52 MP; brightness and capture viewpoint also vary by country. Duplicate groups never cross the primary split.

**Takeaway:** a random held-out test measures average performance; the separate held-out-US test asks whether the detector travels to a new geography.


Capture conditions vary substantially across domains. Norway has much higher-resolution imagery, and countries differ in viewpoint, brightness, contrast, blur, pavement texture, and road markings. The primary duplicate-safe split measures performance on familiar mixed-domain data. A separate held-out-US design measures geographic transfer when the deployment country is absent from training.

# Member 2: a controlled champion–challenger experiment

| Controlled factor | YOLO11n | YOLO11s |
|---|---:|---:|
| Role | Lightweight baseline | Capacity challenger |
| Parameters | 2.58 M | 9.41 M |
| Training manifest | Same hash-pinned 8,000 images | Same hash-pinned 8,000 images |
| Validation / test | Full 5,714 / 5,783 | Full 5,714 / 5,783 |
| Schedule | 30 epochs, 640 px, same seed/augmentation | Identical |

**Metric guide:** mAP@0.50 asks whether damage was found with reasonable overlap; mAP@0.50:0.95 rewards tight localization; recall measures missed damage; F1 balances precision and recall at a validation-selected confidence.

**Takeaway:** the comparison isolates model capacity—data composition, evaluation set, threshold-selection rule and timing hardware remain controlled.


The detection study isolates model capacity through a controlled YOLO11n–YOLO11s comparison. Both runs share the same hash-pinned 8,000-image manifest, seed, augmentation, 640-pixel input, and 30-epoch schedule. Full validation and test sets remain unchanged. With data and evaluation policy held constant, the principal experimental difference is the wider and deeper YOLO11s architecture.

# Limited training compute, full-strength evaluation

| Stage | Images | Why it is used |
|---|---:|---|
| Full duplicate-safe training pool | 26,888 | Available supervised training data |
| Hash-pinned model-training subset | 8,000 | Same representative sample for both detectors |
| Full validation set | 5,714 | Checkpoint and confidence selection |
| Full shared test set | 5,783 | One locked final comparison |

The 8,000-image subset differs from the full training pool by at most **0.01 percentage points** across capture-domain and primary-class composition.

**Takeaway:** compute limits reduce the performance ceiling, but they do not weaken the fairness of the YOLO11n-versus-YOLO11s comparison.

Available Colab compute limited model fitting, not evaluation. Both detectors train on the same representative 8,000-image subset, while the complete 5,714-image validation set selects checkpoints and thresholds and the complete 5,783-image test set supports final reporting. The experiment therefore supports a fair architecture comparison under a shared budget, although it does not estimate the maximum accuracy obtainable from full-data training.

# Detection result: the larger model wins where it matters

<p align="center"><img src="artifacts/presentation/detection_model_comparison.png" alt="YOLO11n versus YOLO11s held-out detection metrics and compute tradeoff" width="980"></p>

| Shared test metric | YOLO11n | YOLO11s |
|---|---:|---:|
| mAP@0.50 | 0.4336 | **0.4439** |
| mAP@0.50:0.95 | 0.2033 | **0.2080** |
| Recall / F1 | 0.4268 / 0.4627 | **0.4447 / 0.4783** |
| D40 AP / recall | 0.2865 / 0.2621 | **0.3209 / 0.3024** |
| Single-image latency (L4) | 16.07 ms | 16.28 ms |

**Takeaway:** YOLO11s is the detection champion because it improves both overall detection and pothole recall with effectively unchanged single-image latency.


YOLO11s leads on every primary shared-test metric. mAP@0.50 increases from 0.4336 to 0.4439, and the stricter mAP@0.50:0.95 also improves. The most relevant class-level change is D40: recall rises from 0.2621 to 0.3024 and average precision from 0.2865 to 0.3209. The gain is modest rather than dramatic, but it is consistent and comes with nearly unchanged single-image L4 latency, supporting YOLO11s as the detection champion.

# Detection gains are concentrated by class

### Per-class AP@0.50 on the shared test set

| Damage class | YOLO11n | YOLO11s | Difference |
|---|---:|---:|---:|
| D00 — longitudinal crack | 0.4412 | **0.4606** | +0.0194 |
| D10 — transverse crack | **0.4330** | 0.4307 | −0.0023 |
| D20 — alligator crack | **0.5738** | 0.5634 | −0.0104 |
| D40 — pothole | 0.2865 | **0.3209** | +0.0344 |

### Validation-selected operating point

| Fixed-threshold test result | YOLO11n | YOLO11s |
|---|---:|---:|
| Confidence threshold | 0.228 | 0.237 |
| True positives / false positives / false negatives | 3,359 / 2,835 / 4,937 | **3,557** / 2,875 / **4,739** |
| Micro precision / recall / F1 | 0.5423 / 0.4049 / 0.4636 | **0.5530 / 0.4288 / 0.4830** |

**Takeaway:** YOLO11s does not improve every class; its champion decision is driven mainly by D00, D40, and stronger fixed-threshold recall/F1.

The per-class table prevents the overall mAP gain from being overinterpreted. D10 and D20 are effectively flat or slightly lower for YOLO11s, while the largest improvement occurs on D40, the rarest and smallest target class. At the validation-selected confidence threshold, YOLO11s produces 198 more true positives and 198 fewer false negatives, with only 40 additional false positives. The resulting micro-F1 increase aligns with the paired-bootstrap result.

# The champion gain is real—but training was compute-limited

<p align="center"><img src="artifacts/presentation/detection_learning_curves.png" alt="YOLO11n and YOLO11s learning curves across 30 epochs" width="900"></p>

| Paired bootstrap micro-F1 | YOLO11n | YOLO11s | Delta (95% CI) |
|---|---:|---:|---:|
| Overall | 0.4636 | 0.4830 | **+0.0194** [+0.0112, +0.0268] |
| D40 | 0.3215 | 0.3630 | **+0.0416** [+0.0158, +0.0688] |

Both primary runs achieve their best validation score at epoch 30.

**Takeaway:** the YOLO11s gain is statistically distinguishable from zero, but both curves suggest the compute-limited training schedule had not fully saturated.


Paired bootstrap analysis tests both models on the same resampled test images. The overall micro-F1 gain is approximately 0.019, with a 95% confidence interval from 0.011 to 0.027. D40 shows a larger gain of about 0.042, again with an interval above zero. Both runs achieve their best validation result at epoch 30, so the comparison is statistically supported while the unsaturated learning curves still justify a longer future schedule.

# Generalization test: unseen geography costs about one sixth of mAP

| Training geography | Strict US slice | Matched non-US slice |
|---|---:|---:|
| All-country 8k | **0.5064** | 0.4247 |
| Non-US 8k | 0.4206 | **0.4391** |

<p align="center"><img src="artifacts/member2/runs/figures/B8_miss_rate_slices.png" alt="Detection miss rates by class, object size, country, blur, and brightness" width="900"></p>

- Removing US training data reduces strict US mAP@0.50 by **16.9% relative**.
- Small-tercile boxes are missed **74.0%** of the time even by YOLO11s.
- D40 remains the worst class: **71.7% miss rate** at the selected operating point.

**Takeaway:** object size is the dominant failure driver, and geography adds a second measurable risk.


Geographic transfer is weaker than the average test score suggests. On the strict US slice, the all-country model reaches 0.5064 mAP@0.50, whereas the model trained without US images reaches 0.4206—a 16.9% relative decrease. The non-US model performs slightly better on the matched non-US slice, indicating specialization rather than universal superiority. Object size remains the larger failure factor: YOLO11s misses 74% of small-tercile targets, and D40 has the highest class miss rate.

# Geographic transfer loss is class-specific

### AP@0.50 on the strict US slice

| Class | All-country YOLO11s | Non-US YOLO11s | Relative change |
|---|---:|---:|---:|
| D00 | **0.6778** | 0.5982 | −11.7% |
| D10 | **0.5647** | 0.4065 | **−28.0%** |
| D20 | **0.5954** | 0.5285 | −11.2% |
| D40 | **0.1877** | 0.1492 | **−20.5%** |
| Overall mAP@0.50 | **0.5064** | 0.4206 | **−16.9%** |

**Takeaway:** transverse cracks and potholes lose the most when US training evidence is removed, so the average domain-shift number hides unequal class risk.

The class breakdown identifies where geographic transfer fails. D10 drops by 28.0% and D40 by 20.5%, substantially more than D00 or D20. Road markings, pavement texture, repair patterns, and capture geometry may affect each category differently. Because the two training manifests are not perfectly matched in country composition, these values are evidence of deployment sensitivity rather than a complete causal explanation of national differences.

# Error analysis: localization and missed small damage dominate

<div style="display:flex;gap:12px;justify-content:center">
  <img src="artifacts/member2/runs/figures/B8_qualitative_panels.png" alt="Dense multi-damage example with ground truth and detector predictions" style="width:49%;height:350px;object-fit:cover;object-position:50% 0%">
  <img src="artifacts/member2/runs/figures/B8_qualitative_panels.png" alt="Small-damage example with ground truth and detector predictions" style="width:49%;height:350px;object-fit:cover;object-position:50% 31%">
</div>
<p align="center"><small>Selected views from the five-case qualitative panel; the complete panel remains in the master notebook.</small></p>

| False-positive type (YOLO11s) | Share | What it means |
|---|---:|---|
| Localization | 43.8% | Correct damage, loose box |
| Background | 33.1% | Shadows, seams, markings, repairs |
| Duplicate | 18.4% | Extra box on an already found target |
| Wrong class | 4.7% | Taxonomy confusion is uncommon |

**Takeaway:** the bottleneck is not learning the four names; it is seeing tiny damage and drawing one tight box around it.


Qualitative review exposes the spatial errors hidden by aggregate metrics. Localization accounts for 43.8% of YOLO11s false positives, followed by background confusion around shadows, seams, markings, and repairs. Duplicate detections occur more often than true class confusion. The model has largely learned the four-category vocabulary; the harder task is finding tiny, low-contrast damage and producing one tight box around each target.

# Member 3: segmentation begins with a sparse-foreground problem

<table><tr>
<td width="50%"><img src="artifacts/member3/eda/m3_fig2_foreground_imbalance.png" alt="Pothole foreground imbalance in Pothole Mix" width="100%"></td>
<td width="50%"><img src="artifacts/member3/eda/m3_fig5_aligned_samples.png" alt="Aligned road images and pothole masks" width="100%"></td>
</tr></table>

- All **4,340** image-mask pairs decode correctly; no dimension mismatch.
- Only **1,184 images (27.3%)** contain pothole pixels.
- Among positive images, the median pothole covers only **2.58%** of the frame.
- Green crack pixels are background for this binary pothole task, creating valid hard negatives.

**Takeaway:** pixel accuracy would look strong by predicting mostly road, so model selection must focus on pothole IoU, Dice, recall and boundaries.


Pothole Mix presents a different imbalance from RDD2022. All 4,340 image–mask pairs decode correctly, but only 1,184 contain pothole pixels. Among positive images, the median pothole occupies just 2.58% of the frame, while many crack-only or normal-road examples act as hard negatives. Background dominance makes ordinary pixel accuracy misleading, so evaluation emphasizes pothole IoU, Dice, recall, and boundary quality.

# Pothole Mix combines six sources with different label prevalence

<p align="center"><img src="artifacts/member3/eda/m3_fig1_split_source_counts.png" alt="Pothole Mix pair counts by split and source component" width="980"></p>

| Source component | Pairs | Pothole-positive | Positive share |
|---|---:|---:|---:|
| CNR road dataset | 20 | 20 | 100% |
| Crack500 | 500 | 0 | 0% |
| Cracks and potholes in road | 2,235 | 564 | 25.2% |
| EdmCrack600 | 600 | 0 | 0% |
| GAPs384 | 385 | 0 | 0% |
| Pothole600 | 600 | 600 | 100% |

**Takeaway:** “Pothole Mix” is not one camera distribution; it mixes pothole-rich sources with three crack-only hard-negative sources.

Source composition explains much of the foreground imbalance. Pothole600 and the small CNR component are entirely positive, while Crack500, EdmCrack600, and GAPs384 contain no red pothole pixels under the binary decoding rule. The largest component contributes both positive and negative road scenes. This mixture improves variety, but source-specific resolution, brightness, and label prevalence also create a domain cue that must be considered when interpreting aggregate segmentation results.

# Two segmentation models, one shared evaluation stack

| | DeepLabV3–MobileNetV3 | SegFormer-B0 |
|---|---|---|
| Core idea | Atrous convolution + multi-scale CNN context | Hierarchical transformer + lightweight MLP decoder |
| Expected strength | Local detail, recall, lower activation memory | Global/multi-scale context, compact parameter count |
| Main risk | Boundary loss and limited long-range context | Upsampling boundaries and lower recall |

**Shared protocol:** official 3,340 / 496 / 504 split, 512×512 input, paired augmentation, cross-entropy + soft-Dice loss, 40 epochs, validation-selected checkpoint and threshold.

**Metric guide:** IoU penalizes both missing and extra mask area; Dice summarizes overlap; recall measures missed pothole pixels; Boundary F1 tests whether the predicted edge follows the true edge.

**Takeaway:** architecture changes, but data, loss, evaluation code and test policy stay fixed.


The segmentation comparison uses one shared protocol for two architecture families. DeepLabV3–MobileNetV3 combines atrous convolution with multi-scale CNN context; SegFormer-B0 combines a hierarchical transformer encoder with a lightweight decoder. Both models use the official split, 512×512 input, paired augmentation, cross-entropy plus soft-Dice loss, 40 epochs, and identical metric definitions. This keeps the comparison focused on architecture rather than pipeline differences.

# Segmentation result: a narrow win with a meaningful tradeoff

<table><tr>
<td width="50%"><img src="artifacts/member3/evaluation/m3_fig7_training_curves.png" alt="DeepLabV3 and SegFormer training curves" width="100%"></td>
<td width="50%"><img src="artifacts/member3/evaluation/m3_fig8_confusion_matrices.png" alt="Segmentation confusion matrices" width="100%"></td>
</tr></table>

| Test metric | DeepLabV3 | SegFormer-B0 |
|---|---:|---:|
| Pothole IoU / Dice | 0.6533 / 0.7903 | **0.6646 / 0.7985** |
| Precision / recall | 0.7889 / **0.7917** | **0.8623** / 0.7436 |
| Boundary F1 | 0.7188 | **0.7334** |
| Parameters | 11.02 M | **3.71 M** |

**Takeaway:** SegFormer wins overlap, precision and boundary quality with 3× fewer parameters; DeepLabV3 remains the recall-oriented alternative.


SegFormer-B0 wins the overall segmentation decision by a narrow margin. Its pothole IoU is 0.6646 and Dice is 0.7985, with stronger precision and Boundary F1 and roughly one third of the parameters. DeepLabV3 retains the higher pothole recall, 0.7917 versus 0.7436. The confusion matrices reflect this tradeoff: SegFormer is more conservative and precise, while DeepLabV3 recovers more pothole pixels at the cost of additional false positives.

# Complete segmentation test scorecard

| Test metric | DeepLabV3–MobileNetV3 | SegFormer-B0 |
|---|---:|---:|
| Pixel accuracy | 0.9862 | **0.9876** |
| Mean IoU | 0.8196 | **0.8260** |
| Background IoU | 0.9858 | **0.9873** |
| Pothole IoU | 0.6533 | **0.6646** |
| Pothole Dice / F1 | 0.7903 | **0.7985** |
| Pothole precision | 0.7889 | **0.8623** |
| Pothole recall | **0.7917** | 0.7436 |
| Mean Boundary F1 | 0.7188 | **0.7334** |
| Inference on NVIDIA T4 | **12.15 ms** | 12.55 ms |
| Parameters | 11.02 M | **3.71 M** |
| Peak GPU memory | **292 MB** | 420 MB |

**Takeaway:** SegFormer leads on overlap, precision, boundaries, and parameter count; DeepLabV3 retains higher recall, slightly lower latency, and lower measured peak memory.

Pixel accuracy and mean IoU are included for completeness but are dominated by the large background class. Pothole-specific IoU, Dice, precision, recall, and Boundary F1 carry more decision value. SegFormer’s parameter advantage does not translate into lower peak activation memory in this run, illustrating why model size and runtime memory should be measured separately. The champion choice favors spatial quality and compact weights, while the recall-oriented alternative remains visible.

# Look beyond the mean: best and worst segmentation cases

<div style="display:flex;gap:12px;justify-content:center">
  <img src="artifacts/member3/evaluation/m3_fig11_qualitative_panels.png" alt="Strong pothole segmentation cases" style="width:49%;height:360px;object-fit:cover;object-position:50% 0%">
  <img src="artifacts/member3/evaluation/m3_fig11_qualitative_panels.png" alt="Difficult pothole segmentation cases" style="width:49%;height:360px;object-fit:cover;object-position:50% 100%">
</div>
<p align="center"><small>Selected strong and difficult cases; the complete comparison panel remains in the master notebook.</small></p>

Close potholes are segmented well (IoU roughly 0.88–0.92), while thin, shallow, distant damage can be missed completely. The per-image IoU distribution is therefore bimodal: a single mean hides a genuine failure cluster.

**Takeaway:** the champion is not “solved”—qualitative failures explain why boundary metrics and human review remain necessary.


Per-image results reveal a failure cluster that the mean score cannot show. Close, clearly visible potholes can reach IoU near 0.9, whereas thin, shallow, distant, or low-contrast damage may be missed completely. Strong interior overlap also does not guarantee an accurate edge. The qualitative panels therefore support reporting Boundary F1 and retaining human review rather than treating the average champion score as a solved task.

# From two models to one human-review application

<table><tr>
<td width="50%"><img src="artifacts/member3/app/CPU-2.png" alt="Verified CPU Gradio road damage analysis" width="100%"></td>
<td width="50%"><img src="artifacts/member3/app/GPU.png" alt="Verified GPU Gradio road damage analysis" width="100%"></td>
</tr></table>

| Evidence visible to the reviewer | Purpose |
|---|---|
| Detection boxes, classes, and confidences | Show what was detected and where |
| Segmentation overlay and damaged-area percentage | Show the estimated pothole footprint |
| Model, device, and latency metadata | Make the inference context auditable |
| Low/Medium/High prototype priority and reasons | Explain the triage suggestion |
| Human-review warning | Preserve the decision boundary |

The captured CUDA run completed the full YOLO11s + SegFormer path in **56 ms**; CPU first-call latency exceeded one second. These app timings include end-to-end overhead and are separate from controlled per-model benchmarks.

**Takeaway:** the interface exposes evidence and limitations; its priority is a triage suggestion, not a civil-engineering rating.

The Gradio prototype combines both champions in one review surface. It displays YOLO boxes and confidences, the SegFormer mask and damaged-area percentage, model metadata, latency, and a transparent Low/Medium/High prototype priority with reasons. The captured CUDA path completes both models in 56 ms, while CPU first-call latency exceeds one second. These are end-to-end application measurements, and the displayed priority remains a triage suggestion rather than a certified road-condition rating.

# Final scorecard: champion does not mean universally best

<p align="center"><img src="artifacts/final/model_tradeoff_summary.svg" alt="Detection and segmentation champion-challenger tradeoff summary" width="1100"></p>

**Takeaway:** both champions win narrowly. YOLO11s earns deployment preference through significant pothole-recall gains; SegFormer wins overlap and size, but DeepLabV3 is defensible when missed potholes cost more than false alarms.


The final selections reflect deployment objectives rather than a universal winner. YOLO11s is preferred because its statistically supported gain includes better pothole recall with nearly unchanged single-image latency. SegFormer-B0 is preferred for overlap, boundary quality, and parameter efficiency. YOLO11n and DeepLabV3 remain documented alternatives when compute limits or the relative cost of misses and false alarms changes.

# Accuracy and resource cost must be read together

| Task | Model | Hardware | Parameters | Compute / memory | Single-image inference |
|---|---|---|---:|---:|---:|
| Detection | YOLO11n | NVIDIA L4 | **2.58 M** | **6.38 GFLOPs / 0.10 GB** | **16.07 ms** |
| Detection | YOLO11s | NVIDIA L4 | 9.41 M | 21.43 GFLOPs / 0.15 GB | 16.28 ms |
| Segmentation | DeepLabV3–MobileNetV3 | NVIDIA T4 | 11.02 M | **292 MB peak** | **12.15 ms** |
| Segmentation | SegFormer-B0 | NVIDIA T4 | **3.71 M** | 420 MB peak | 12.55 ms |

*Detection and segmentation latencies come from different hardware and task pipelines, so they should be compared only within each task.*

**Takeaway:** parameter count, FLOPs, activation memory, and latency describe different costs; no single efficiency number selects both champions.

YOLO11s uses roughly 3.6 times as many parameters and 3.4 times the reported FLOPs of YOLO11n, yet single-image latency remains nearly identical on the L4 benchmark. SegFormer has about one third of DeepLabV3’s parameters but higher measured peak GPU memory, while latency differs by only 0.40 ms on T4. These results favor task-specific deployment profiling rather than assuming that smaller weights always imply lower runtime cost.

# Deployment must be observable, versioned, and reversible

<p align="center"><img src="artifacts/final/deployment_architecture.svg" alt="Versioned road damage deployment architecture with monitoring and rollback" width="1080"></p>

**Operational loop:** monitor → label reviewed failures → train a versioned challenger → gate on locked tests → canary → promote or roll back.

**Takeaway:** model operations preserve the experiment’s central principle: version everything, protect the test set, measure drift, and keep a previous champion ready.


Operational deployment requires a versioned safety loop around the models. Data, preprocessing, checkpoints, thresholds, and review policy should move together as one release. Monitoring covers input drift, confidence, latency, and reviewer-confirmed failures. Locked evaluation gates, canary promotion, and a retained previous champion make updates observable and reversible rather than silently replacing one behavior with another.

# The next experiments should target observed failures

| Priority | Next experiment | Evidence of improvement |
|---|---|---|
| Small damage | Higher-resolution crops or tiled detection | Better small-tercile and D40 recall |
| Geographic transfer | Add labeled countries and camera types | Smaller held-out-domain performance drop |
| Training ceiling | Longer full-data schedules | Validation curves that clearly plateau |
| Reliability | Confidence calibration and review study | Better agreement between confidence and correctness |
| Pothole boundaries | Boundary-aware loss or refinement | Higher Boundary F1 without recall collapse |
| Environmental coverage | Night, rain, snow, glare, and water tests | Documented acceptance ranges and failure triggers |

**Takeaway:** future work is prioritized by measured errors, not by adding a more complicated model without a clear failure target.

Observed failures define the next research priorities. Small-object detection comes first because it is the largest measured source of misses, followed by broader geographic and environmental coverage. Longer schedules can test whether the YOLO curves continue improving, while calibration and reviewer studies can assess whether confidence values support real decisions. Boundary-aware segmentation methods should improve edge quality without sacrificing the recall already achieved by the current models.

# Conclusion: decision support, not autonomous maintenance

### What remains hard

- **Small-object recall:** YOLO11s still misses 74% of small-tercile boxes; D40 recall is 0.3024.
- **Domain shift:** removing US training data costs 16.9% relative mAP on strict US images.
- **Localization and boundaries:** detection mAP@0.50:0.95 is about 0.21; segmentation Boundary F1 is about 0.73.
- **Coverage:** night, snow, water, glare, unusual cameras and new jurisdictions are not acceptance-tested.

### Final answer to the opening questions

- **What broke and where?** YOLO11s provides four-class boxes.
- **How much area?** SegFormer-B0 provides the pothole mask and footprint.
- **Who decides?** A human inspector, with evidence and explicit uncertainty.

> The strongest result is not automation; it is an auditable way to prioritize road images without hiding where the models fail.

# Questions?


The completed system provides visible evidence for the three opening questions: YOLO11s identifies damage type and approximate location, and SegFormer-B0 estimates the pothole footprint. Small targets, unfamiliar geography, tight localization, difficult boundaries, and untested weather conditions remain important limitations. The defensible project outcome is an auditable image-triage workflow in which a human inspector retains authority over maintenance decisions.

# Appendix — team roles and sources

| Member | Presentation ownership |
|---|---|
| **Member 1 — Austin Wang** | Problem framing, RDD2022 data engineering, EDA, split design, and combined scorecard |
| **Member 2 — Kevin Fan** | Detection experiment, results, generalization, error analysis, and deployment operations |
| **Member 3 — RJ Xia** | Segmentation data, models, results, qualitative failures, application, and conclusion |

## Key sources

1. Arya, D. et al. RDD2022 dataset and paper: https://doi.org/10.6084/m9.figshare.21431547 and https://doi.org/10.1002/gdj3.260 (CC BY 4.0).
2. Pothole Mix v1.0: https://doi.org/10.17632/kfth5g2xk3.2; component-license record in `artifacts/member3/pothole_mix_provenance.json`.
3. Ultralytics YOLO11: https://docs.ultralytics.com/models/yolo11.
4. Chen, L.-C. et al., DeepLabV3: https://arxiv.org/abs/1706.05587; Howard, A. et al., MobileNetV3: https://arxiv.org/abs/1905.02244.
5. Xie, E. et al., SegFormer: https://arxiv.org/abs/2105.15203.
6. Gradio: https://www.gradio.app/docs.
7. Opening photograph: Prosthetic Head, “Marked Pothole,” Wikimedia Commons, CC BY-SA 4.0: https://commons.wikimedia.org/wiki/File:Marked_Pothole.jpg.

Exact methods, complete tables, code, saved outputs and the full reference list remain in `Road_Damage_Final_Project_Master_New.ipynb`.
